In [ ]:
import ollama
import re

In [ ]:
response = ollama.chat(
    model = "llama3.2",
    messages = [
        {
            "role":"user",
            "content":"How did Erling Haaland perform the the 2022 FIFA " + 
                "World Cup?"
        }
    ],
    stream = True
)

In [ ]:
type(response)

In [ ]:
print(response["message"]["content"])

Stream test

In [ ]:
stream = ollama.chat(
    model="llama3.2",
    messages=[
        {
            "role":"user",
            "content":"How did Erling Haaland perform the the 2022 FIFA " + 
                "World Cup?"
        }
    ],
    stream=True
)



for chunk in stream:
    print(chunk["message"]["content"], end="", flush=True)
print()

Create a way to chat with the bot

In [ ]:
def Prompt_User():
    user_prompt = input("\nPrompt")
    
    # Check if user wants to exit the conversation
    if user_prompt.strip().lower() == "exit":
        return "break"
    
    # Skip answering if input is empty
    if not user_prompt.strip():
        return "continue"
    
    return user_prompt

In [ ]:
def Print_Response(Stream, Max_Row_Len):
    full_response = ""
    row_len = 0
    
    for chunk in Stream:
        text_chunk = chunk["message"]["content"]
        
        # Update length of row
        if "\n" in text_chunk:
            row_len = 0
        else:
            row_len += len(text_chunk)
        
        # Break line on space if row length is too long
        if row_len >= Max_Row_Len:
            text_chunk = re.sub(" ", "\n", text_chunk, count = 1)
            row_len = 0
        
        # Print the text chunk and save the full response
        print(text_chunk, end = "", flush=True)
        full_response += text_chunk
    
    print("\n")
    
    return full_response

In [ ]:
# The chat bot

max_row_len = 120

# Create a list to store the chat history
messages = [
    {
        "role" : "system",
        "content" : "You are knowledge bank for football facts. Answer " +
            "briefly and stick to the facts. Do not speculate, only answer " +
            "with what you know"
    }
]

print("Start the chat with Llama 3.2. Write 'exit' to leave the chat")

# Create the loop that starts the chat
while True:
    # Gather input
    user_prompt = Prompt_User()
    match user_prompt:
        case "break":
            break
        case "continue":
            continue
        case _:
            print("You: " + user_prompt + "\n")
        
    
    # Save prompt to messages history
    messages.append({"role" : "user", "content" : user_prompt})
    
    print("AI: ", end= "", flush=True)
    
    # Send the prompt to the model
    stream = ollama.chat(
        model = "llama3.2",
        messages = messages,
        stream = True
    )
    
    # Print the message generated by the model
    full_response = Print_Response(Stream=stream, Max_Row_Len=max_row_len)
    
    # Save the answer to messages history
    messages.append({"role" : "assistant", "content" : full_response})


Test ChatBot() Class

In [ ]:
import sys
sys.path.append('../')

from src.generation import ChatBot

In [ ]:
chat_bot = ChatBot(
    model_name = 'llama3.2',
    system_instructions = "You are knowledge bank for football facts. Answer " +
        "briefly and stick to the facts. Do not speculate, only answer " +
        "with what you know"
    )

while True:
    user_prompt = input("\nPrompt: ")
    
    if user_prompt.strip().lower() == "exit":
        break
    
    response = chat_bot.ask(user_prompt, stream = True)
    
    print()
    for chunk in response:
        print(chunk, end="", flush=True)
    print()